In [1]:
import os
os.environ["HF_HOME"] = "/home/yandex/APDL2425a/group_12/gorodissky/.cache/huggingface"
print(f"HF_HOME set to:\t\t {os.environ['HF_HOME']}")

import torch
print(f"CUDA available: \t{torch.cuda.is_available()}")
print(f"Torch version: \t\t{torch.__version__}")
if torch.cuda.is_available():
    print(f"Number of CUDA devices\t {torch.cuda.device_count()}")
    print(f"CUDA device:\t\t {torch.cuda.get_device_name(torch.cuda.current_device())}")

HF_HOME set to:		 /home/yandex/APDL2425a/group_12/gorodissky/.cache/huggingface
CUDA available: 	True
Torch version: 		2.7.1+cu126
Number of CUDA devices	 4
CUDA device:		 NVIDIA GeForce GTX TITAN X


In [2]:
from adais.datasets import mmlu_pro
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from pathlib import Path
from sae.general import stop_expr
from adais.adaptive.probe import CorrectnessScorer

2026-02-19 18:21:48.666069: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
from sae.general import get_latest_cpt


base_path = "/home/yandex/APDL2425a/group_12/gorodissky/sae/data/qa/MMLU"
model_small = "Qwen/Qwen2.5-3B-Instruct"
latest_cpt_small = get_latest_cpt(f"{base_path}/{model_small}")
model_big = "Qwen/Qwen2.5-14B-Instruct"
latest_cpt_big = get_latest_cpt(f"{base_path}/{model_big}")
qa_df_small = pd.read_pickle(f"{base_path}/{model_small}/{latest_cpt_small}/qa_dataset.pkl")
qa_df_big = pd.read_pickle(f"{base_path}/{model_big}/{latest_cpt_big}/qa_dataset.pkl")

In [8]:
(qa_df_small["question_id"] == qa_df_big["question_id"]).all()

True

In [9]:
df_new = pd.DataFrame({
    "question_id": qa_df_small["question_id"],
    "question": qa_df_small["question"],
    "answer_small": qa_df_small["answer"],
    "is_correct_small": qa_df_small["is_correct"],
    "answer_big": qa_df_big["answer"],
    "is_correct_big": qa_df_big["is_correct"],
    "golden_label": qa_df_small["golden_label"],
    "activation_mid_small": qa_df_small["activation_mid"],
    "activation_pre_mid": qa_df_small["activation_pre_mid"],
    "activation_post_mid": qa_df_small["activation_post_mid"],
})

In [10]:
print(df_new["is_correct_small"].value_counts())
print(df_new["is_correct_big"].value_counts())

is_correct_small
False    190
True     130
Name: count, dtype: int64
is_correct_big
True     191
False    129
Name: count, dtype: int64


In [11]:
table_count = pd.DataFrame(index=["small_correct", "small_wrong"],
                     columns=["big_correct", "big_wrong"])
table_count.loc["small_correct", "big_correct"] = ((df_new["is_correct_small"]) & (df_new["is_correct_big"])).sum()
table_count.loc["small_correct", "big_wrong"] = ((df_new["is_correct_small"]) & (~df_new["is_correct_big"])).sum()
table_count.loc["small_wrong", "big_correct"] = ((~df_new["is_correct_small"]) & (df_new["is_correct_big"])).sum()
table_count.loc["small_wrong", "big_wrong"] = ((~df_new["is_correct_small"]) & (~df_new["is_correct_big"])).sum()
print("Count of questions by correctness of small and big models:")
print(table_count)

Count of questions by correctness of small and big models:
              big_correct big_wrong
small_correct         109        21
small_wrong            82       108


In [ ]:
def avg_activation(row):
    return (row["activation_pre_mid"] + row["activation_mid_small"] + row["activation_post_mid"]) / 3.0
df_new["activation_avg"] = df_new.apply(avg_activation, axis=1)


In [ ]:
probe = CorrectnessScorer.load("/home/yandex/APDL2425a/group_12/gorodissky/AdaIS/output/probe_results/MMLU/Qwen/Qwen2.5-3B-Instruct/2026-02-16_17:06/correctness_scorer.npz")
df_new["probe_score"] = df_new["activation_avg"].apply(lambda x: probe.score(np.expand_dims(x, axis=0)))
table_score_stats = pd.DataFrame(index=["small_correct", "small_wrong"],
                     columns=["big_correct", "big_wrong"])

def mean_std(mask):
    s = df_new.loc[mask, "probe_score"]
    return f"{s.mean():.4f} ± {s.std():.4f}"

table_score_stats.loc["small_correct", "big_correct"] = mean_std(df_new["is_correct_small"] & df_new["is_correct_big"])
table_score_stats.loc["small_correct", "big_wrong"]   = mean_std(df_new["is_correct_small"] & ~df_new["is_correct_big"])
table_score_stats.loc["small_wrong", "big_correct"]   = mean_std(~df_new["is_correct_small"] & df_new["is_correct_big"])
table_score_stats.loc["small_wrong", "big_wrong"]     = mean_std(~df_new["is_correct_small"] & ~df_new["is_correct_big"])

print("Probe score (mean ± std) by correctness of small and big models:")
print(table_score_stats)

In [ ]:

# compute distances for each activation (using activation_avg)
def is_closer_to_correct(act):
    d_correct = np.linalg.norm(act - probe.correct_centroid)
    d_incorrect = np.linalg.norm(act - probe.incorrect_centroid)
    return d_correct < d_incorrect
df_new["is_closer_correct"] = df_new["activation_avg"].apply(is_closer_to_correct)

table_is_closer = pd.DataFrame(index=["small_correct", "small_wrong"],
                     columns=["big_correct", "big_wrong"])

def closer_percentage(mask):
    subset = df_new.loc[mask, "is_closer_correct"]
    if len(subset) == 0:
        return "N/A"
    return f"{subset.mean()*100:.2f}%"

table_is_closer.loc["small_correct", "big_correct"] = closer_percentage(df_new["is_correct_small"] & df_new["is_correct_big"])
table_is_closer.loc["small_correct", "big_wrong"]   = closer_percentage(df_new["is_correct_small"] & ~df_new["is_correct_big"])
table_is_closer.loc["small_wrong", "big_correct"]   = closer_percentage(~df_new["is_correct_small"] & df_new["is_correct_big"])
table_is_closer.loc["small_wrong", "big_wrong"]     = closer_percentage(~df_new["is_correct_small"] & ~df_new["is_correct_big"])

print("Percentage of questions where activation is closer to correct centroid by correctness of small and big models:")
print(table_is_closer)

In [3]:
from sae.general import get_latest_cpt
base_path = "/home/yandex/APDL2425a/group_12/gorodissky/sae/data/qa/MMLU"
model_names = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-14B-Instruct",
    "Qwen/Qwen2.5-32B-Instruct",
]
for model_name in model_names:
    last_cpt = get_latest_cpt(f"{base_path}/{model_name}")
    df = pd.read_pickle(f"{base_path}/{model_name}/{last_cpt}/qa_dataset.pkl")
    print(f"Model: {model_name}, latest checkpoint: {last_cpt}")
    print(df["is_correct"].value_counts())


Model: Qwen/Qwen2.5-0.5B-Instruct, latest checkpoint: 2026_02_19-14:07
is_correct
False    13
True      3
Name: count, dtype: int64
Model: Qwen/Qwen2.5-3B-Instruct, latest checkpoint: 2026_02_19-14:28
is_correct
False    190
True     130
Name: count, dtype: int64
Model: Qwen/Qwen2.5-14B-Instruct, latest checkpoint: 2026_02_19-14:55
is_correct
True     191
False    129
Name: count, dtype: int64
Model: Qwen/Qwen2.5-32B-Instruct, latest checkpoint: 2026_02_19-17:24
is_correct
True     215
False    105
Name: count, dtype: int64


In [ ]:
df["is_correct"].value_counts()

In [ ]:
for res, ans, golden_label in df.loc[:, ["response", "answer", "golden_label"]].values:
    print(f"Response:\n{res}\nAnswer:\n{ans}\nGolden label:\n{golden_label}")
    print("="*10000)

In [ ]:
qwen = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", dtype="auto", device_map="auto")
print(qwen)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
question = "What is the capital of France?"
prompt = [
    {
        "role": "user",
        "content": question
    },
]
input_ids = tokenizer.apply_chat_template(
    prompt, 
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt")

print(input_ids)

In [ ]:
for _ in range(10):
    output_ids = qwen.generate(**input_ids)
    print(tokenizer.batch_decode(output_ids, skip_special_tokens=False)[0])
    print("="*1000)